In [6]:
!pip install transformers openai-clip -q

TRAIN_CSV       = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/train/train.csv" 
EVAL_CSV        = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/eval/val.csv"
TEST_CSV        = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/test_release/index_text.csv"
TRAIN_IMAGE_DIR = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/train/train_images"  
EVAL_IMAGE_DIR  = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/eval/val_no_labels/val_images"
TEST_IMAGE_DIR  = "/kaggle/input/datasets/sumaiyazaman110/vaxmeme/Dataset/test_release/test_images"
OUTPUT_DIR      = "/kaggle/working/"


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 2 — Imports
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
import os, random, zipfile
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import f1_score, classification_report

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
import clip

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 3 — Shared Utilities (Focal Loss, seed)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0):
        super().__init__()
        self.gamma = gamma
    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, reduction="none")
        pt  = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 4 — MODEL A: Text-Only DeBERTa
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

TEXT_MODEL  = "bert-base-uncased"
MAX_LEN     = 128
BATCH_SIZE  = 16
EPOCHS_TEXT = 5
TEXT_LR     = 2e-5   # bert-base-uncased standard LR

class TextDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test   = is_test

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row.get("post_text") or "")
        enc  = self.tokenizer(text, max_length=MAX_LEN, padding="max_length",
                              truncation=True, return_tensors="pt")
        item = {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "token_type_ids": enc["token_type_ids"].squeeze(0),
        }
        if not self.is_test:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

class TextClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder    = AutoModel.from_pretrained(TEXT_MODEL)
        hidden          = self.encoder.config.hidden_size
        self.dropout    = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, 3)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                           token_type_ids=token_type_ids)
        cls_feat  = out.last_hidden_state[:, 0, :]
        mask      = attention_mask.unsqueeze(-1).float()
        mean_feat = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(1e-9)
        pooled    = (cls_feat + mean_feat) / 2
        return self.classifier(self.dropout(pooled))

def train_text_model():
    print("\n" + "="*55)
    print("STEP 1/3 — Training Text-Only DeBERTa")
    print("="*55)

    train_df = pd.read_csv(TRAIN_CSV)
    eval_df  = pd.read_csv(EVAL_CSV)
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)

    train_loader = DataLoader(TextDataset(train_df, tokenizer),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    eval_loader  = DataLoader(TextDataset(eval_df, tokenizer),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model     = TextClassifier().to(DEVICE)
    criterion = FocalLoss(gamma=2.0)
    scaler    = GradScaler()
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(),    "lr": TEXT_LR},
        {"params": model.classifier.parameters(), "lr": TEXT_LR * 10},
    ], weight_decay=1e-2)

    total_steps  = len(train_loader) * EPOCHS_TEXT
    scheduler    = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * 0.1), total_steps
    )

    best_f1, best_path = 0, os.path.join(OUTPUT_DIR, "best_text.pt")

    for epoch in range(1, EPOCHS_TEXT + 1):
        # Train
        model.train()
        for batch in train_loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            ttids = batch["token_type_ids"].to(DEVICE)
            lbls  = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(ids, mask, ttids), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

        # Eval
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for batch in eval_loader:
                ids   = batch["input_ids"].to(DEVICE)
                mask  = batch["attention_mask"].to(DEVICE)
                ttids = batch["token_type_ids"].to(DEVICE)
                logits = model(ids, mask, ttids)
                preds.extend(logits.argmax(-1).cpu().numpy())
                labs.extend(batch["label"].numpy())

        val_f1 = f1_score(labs, preds, average="macro")
        print(f"  Epoch {epoch}/{EPOCHS_TEXT}  Val Macro F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ Saved (F1: {best_f1:.4f})")

    # Per-class report on eval
    print("\n  Final per-class breakdown:")
    print(classification_report(labs, preds,
          target_names=["Vaccine Critical", "Neutral", "Pro-vaccine"], digits=4))

    # Generate test logits
    test_df     = pd.read_csv(TEST_CSV)
    test_loader = DataLoader(TextDataset(test_df, tokenizer, is_test=True),
                             batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.load_state_dict(torch.load(best_path))
    model.eval()

    all_logits = []
    with torch.no_grad():
        for batch in test_loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            ttids = batch["token_type_ids"].to(DEVICE)
            logits = model(ids, mask, ttids)
            all_logits.append(logits.cpu().float().numpy())

    text_logits = np.concatenate(all_logits, axis=0)
    np.save(os.path.join(OUTPUT_DIR, "logits_text.npy"), text_logits)
    print(f"\n  Text logits saved → logits_text.npy  shape: {text_logits.shape}")
    print(f"  Best Text-Only Val F1: {best_f1:.4f}")
    return text_logits


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 5 — MODEL B: Multimodal DeBERTa + CLIP
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

CLIP_MODEL       = "ViT-B/32"
FUSION_DIM       = 512
CLIP_HIDDEN      = 512
TEXT_HIDDEN      = 768
EPOCHS_MM        = 6
CLIP_UNFREEZE_AT = 3    # epoch to unfreeze CLIP

class MemeDataset(Dataset):
    def __init__(self, df, tokenizer, clip_prep, image_dir, is_test=False):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.clip_prep = clip_prep
        self.image_dir = image_dir   # ← passed in per split
        self.is_test   = is_test
        self._blank    = torch.zeros(3, 224, 224)

    def __len__(self): return len(self.df)

    def _load_img(self, fname):
        try:
            return self.clip_prep(Image.open(
                os.path.join(self.image_dir, str(fname))).convert("RGB"))
        except:
            return self._blank.clone()

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        text = str(row.get("post_text") or "")
        enc  = self.tokenizer(text, max_length=MAX_LEN, padding="max_length",
                              truncation=True, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["image"] = self._load_img(row["index"])
        if not self.is_test:
            item["label"] = torch.tensor(int(row["label"]), dtype=torch.long)
        return item

class GatedMultimodalModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Text encoder
        self.text_enc = AutoModel.from_pretrained(TEXT_MODEL)
        # Image encoder (CLIP visual only)
        clip_mdl, _   = clip.load(CLIP_MODEL, device="cpu")
        self.img_enc  = clip_mdl.visual.float()
        for p in self.img_enc.parameters():   # frozen initially
            p.requires_grad = False
        # Projections
        self.text_proj = nn.Sequential(
            nn.Linear(TEXT_HIDDEN, FUSION_DIM), nn.LayerNorm(FUSION_DIM),
            nn.GELU(), nn.Dropout(0.1))
        self.img_proj  = nn.Sequential(
            nn.Linear(CLIP_HIDDEN, FUSION_DIM), nn.LayerNorm(FUSION_DIM),
            nn.GELU(), nn.Dropout(0.1))
        # Cross-attention: text attends to image
        self.cross_attn = nn.MultiheadAttention(
            FUSION_DIM, num_heads=8, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(FUSION_DIM)
        # Gate: decides how much to trust image vs text
        self.gate = nn.Sequential(
            nn.Linear(FUSION_DIM * 2, FUSION_DIM), nn.ReLU(),
            nn.Linear(FUSION_DIM, 1), nn.Sigmoid())
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(FUSION_DIM, 256), nn.GELU(),
            nn.Dropout(0.1), nn.Linear(256, 3))

    def unfreeze_clip(self):
        for p in self.img_enc.parameters():
            p.requires_grad = True
        print("  → CLIP unfrozen")

    def forward(self, input_ids, attention_mask, image, token_type_ids=None):
        # Text
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out      = self.text_enc(**kwargs)
        cls_f    = out.last_hidden_state[:, 0, :]
        mask     = attention_mask.unsqueeze(-1).float()
        mean_f   = (out.last_hidden_state * mask).sum(1) / mask.sum(1).clamp(1e-9)
        text_f   = self.text_proj((cls_f + mean_f) / 2)

        # Image
        grad_fn = torch.enable_grad if any(
            p.requires_grad for p in self.img_enc.parameters()) else torch.no_grad
        with grad_fn():
            img_f = self.img_proj(self.img_enc(image).float())

        # Cross-attention (text queries image)
        q = text_f.unsqueeze(1)
        k = img_f.unsqueeze(1)
        cross, _ = self.cross_attn(q, k, k)
        cross    = self.norm(cross.squeeze(1))

        # Gating
        gate  = self.gate(torch.cat([text_f, cross], dim=-1))
        fused = gate * cross + (1 - gate) * text_f

        return self.classifier(fused)

def train_multimodal_model():
    print("\n" + "="*55)
    print("STEP 2/3 — Training Multimodal DeBERTa + CLIP")
    print("="*55)

    train_df  = pd.read_csv(TRAIN_CSV)
    eval_df   = pd.read_csv(EVAL_CSV)
    tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL)
    _, clip_prep = clip.load(CLIP_MODEL, device="cpu")

    train_loader = DataLoader(MemeDataset(train_df, tokenizer, clip_prep, TRAIN_IMAGE_DIR),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    eval_loader  = DataLoader(MemeDataset(eval_df, tokenizer, clip_prep, EVAL_IMAGE_DIR),
                              batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model     = GatedMultimodalModel().to(DEVICE)
    criterion = FocalLoss(gamma=2.0)
    scaler    = GradScaler()
    optimizer = torch.optim.AdamW([
        {"params": model.text_enc.parameters(),   "lr": 1e-5},
        {"params": model.img_enc.parameters(),    "lr": 5e-7},
        {"params": model.text_proj.parameters(),  "lr": 5e-5},
        {"params": model.img_proj.parameters(),   "lr": 5e-5},
        {"params": model.cross_attn.parameters(), "lr": 5e-5},
        {"params": model.gate.parameters(),       "lr": 5e-5},
        {"params": model.classifier.parameters(), "lr": 5e-5},
    ], weight_decay=1e-2)

    total_steps = len(train_loader) * EPOCHS_MM
    scheduler   = get_cosine_schedule_with_warmup(
        optimizer, int(total_steps * 0.1), total_steps)

    best_f1, best_path = 0, os.path.join(OUTPUT_DIR, "best_multimodal.pt")

    for epoch in range(1, EPOCHS_MM + 1):
        if epoch == CLIP_UNFREEZE_AT + 1:
            model.unfreeze_clip()

        # Train
        model.train()
        for batch in train_loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            img   = batch["image"].to(DEVICE)
            ttids = batch.get("token_type_ids")
            if ttids is not None: ttids = ttids.to(DEVICE)
            lbls  = batch["label"].to(DEVICE)

            optimizer.zero_grad()
            with autocast():
                loss = criterion(model(ids, mask, img, ttids), lbls)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()

        # Eval
        model.eval()
        preds, labs = [], []
        with torch.no_grad():
            for batch in eval_loader:
                ids   = batch["input_ids"].to(DEVICE)
                mask  = batch["attention_mask"].to(DEVICE)
                img   = batch["image"].to(DEVICE)
                ttids = batch.get("token_type_ids")
                if ttids is not None: ttids = ttids.to(DEVICE)
                logits = model(ids, mask, img, ttids)
                preds.extend(logits.argmax(-1).cpu().numpy())
                labs.extend(batch["label"].numpy())

        val_f1 = f1_score(labs, preds, average="macro")
        print(f"  Epoch {epoch}/{EPOCHS_MM}  Val Macro F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), best_path)
            print(f"  ✓ Saved (F1: {best_f1:.4f})")

    print("\n  Final per-class breakdown:")
    print(classification_report(labs, preds,
          target_names=["Vaccine Critical", "Neutral", "Pro-vaccine"], digits=4))

    # Generate test logits
    test_df     = pd.read_csv(TEST_CSV)
    test_loader = DataLoader(
        MemeDataset(test_df, tokenizer, clip_prep, TEST_IMAGE_DIR, is_test=True),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.load_state_dict(torch.load(best_path))
    model.eval()

    all_logits = []
    with torch.no_grad():
        for batch in test_loader:
            ids   = batch["input_ids"].to(DEVICE)
            mask  = batch["attention_mask"].to(DEVICE)
            img   = batch["image"].to(DEVICE)
            ttids = batch.get("token_type_ids")
            if ttids is not None: ttids = ttids.to(DEVICE)
            with autocast():
                logits = model(ids, mask, img, ttids)
            all_logits.append(logits.cpu().float().numpy())

    mm_logits = np.concatenate(all_logits, axis=0)
    np.save(os.path.join(OUTPUT_DIR, "logits_multimodal.npy"), mm_logits)
    print(f"\n  Multimodal logits saved → logits_multimodal.npy  shape: {mm_logits.shape}")
    print(f"  Best Multimodal Val F1: {best_f1:.4f}")
    return mm_logits


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 6 — STEP 3: Ensemble both models → Final submission
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

def make_final_submission(text_logits, mm_logits):
    print("\n" + "="*55)
    print("STEP 3/3 — Ensembling & Creating Submission")
    print("="*55)

    test_df = pd.read_csv(TEST_CSV)

    # Softmax → probabilities, then weighted average
    text_probs = torch.softmax(torch.tensor(text_logits), dim=-1).numpy()
    mm_probs   = torch.softmax(torch.tensor(mm_logits),   dim=-1).numpy()

    # Weight: 50/50 — adjust if one model has much better eval F1
    final_probs = 0.5 * text_probs + 0.5 * mm_probs
    preds       = final_probs.argmax(axis=1)

    sub_df = pd.DataFrame({
        "index": test_df["index"].values,
        "label": preds
    }).sort_values("index").reset_index(drop=True)

    csv_path = os.path.join(OUTPUT_DIR, "predictions.csv")
    zip_path = os.path.join(OUTPUT_DIR, "submission_final.zip")
    sub_df.to_csv(csv_path, index=False)
    with zipfile.ZipFile(zip_path, "w") as zf:
        zf.write(csv_path, "predictions.csv")

    print(f"\n  Label distribution: {dict(pd.Series(preds).value_counts().sort_index())}")
    print(f"\n  ✅ DONE! Submit this file → {zip_path}")


# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# CELL 7 — RUN EVERYTHING
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
text_logits = train_text_model()
mm_logits   = train_multimodal_model()
make_final_submission(text_logits, mm_logits)

Device: cuda

STEP 1/3 — Training Text-Only DeBERTa


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

  Epoch 5/6  Val Macro F1: 0.7967


/tmp/ipykernel_55/860231040.py:353: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


  Epoch 6/6  Val Macro F1: 0.7924

  Final per-class breakdown:
                  precision    recall  f1-score   support

Vaccine Critical     0.8135    0.8214    0.8174       308
         Neutral     0.7417    0.6850    0.7122       327
     Pro-vaccine     0.8248    0.8715    0.8475       389

        accuracy                         0.7969      1024
       macro avg     0.7933    0.7926    0.7924      1024
    weighted avg     0.7949    0.7969    0.7953      1024



/tmp/ipykernel_55/860231040.py:402: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(



  Multimodal logits saved → logits_multimodal.npy  shape: (1025, 3)
  Best Multimodal Val F1: 0.8092

STEP 3/3 — Ensembling & Creating Submission

  Label distribution: {0: np.int64(310), 1: np.int64(381), 2: np.int64(334)}

  ✅ DONE! Submit this file → /kaggle/working/submission_final.zip
